# AutoRouteAI — Routing Demo

This notebook demonstrates the end-to-end routing pipeline for Chennai.

**Steps:**
1. Build graph from OSM data
2. Run A* routing between two points
3. Apply AI re-ranking
4. Visualise routes on a map


In [ ]:
import sys
sys.path.insert(0, '../backend')

import folium
from routing.graph_builder import build_graph, get_graph
from routing.astar import astar_route
from routing.reranker import rerank_routes
from utils.geo import coords_to_node, route_geometry
from ai.route_ranker import RouteRanker
from ai.traffic_predictor import TrafficPredictor

print('Imports OK')

In [ ]:
# Build or load graph
# Uncomment to build from scratch (takes ~2 min)
# G = build_graph('chennai')

G = get_graph('chennai')
if G is None:
    raise RuntimeError('Graph not found. Run: python scripts/build_graph.py --city chennai')

print(f'Nodes: {len(G.nodes):,}')
print(f'Edges: {len(G.edges):,}')

In [ ]:
# Define origin and destination
ORIGIN = {'lat': 13.0827, 'lng': 80.2707}       # Chennai Central
DEST   = {'lat': 13.0604, 'lng': 80.2496}       # T. Nagar

origin_node = coords_to_node(G, ORIGIN['lat'], ORIGIN['lng'])
dest_node   = coords_to_node(G, DEST['lat'], DEST['lng'])
print(f'Origin node: {origin_node}')
print(f'Destination node: {dest_node}')

In [ ]:
# Generate candidate routes
from types import SimpleNamespace
preferences = SimpleNamespace(
    avoid_floods=True,
    road_quality_weight=0.3,
    time_weight=0.4,
    driver_score_weight=0.3,
)

candidates = astar_route(G, origin_node, dest_node, k=5, preferences=preferences)
print(f'Found {len(candidates)} candidate routes')
for i, r in enumerate(candidates):
    print(f'  Route {i+1}: {r["distance_km"]} km, {r["eta_minutes"]} min, road_score={r["road_quality_score"]:.2f}')

In [ ]:
import asyncio

ranker = RouteRanker()
traffic = TrafficPredictor()

ranked = asyncio.run(rerank_routes(candidates, G, ranker, traffic, preferences))

print('Ranked routes:')
for i, r in enumerate(ranked[:3]):
    label = ['Driver Recommended', 'Fastest', 'Alternative'][i]
    print(f'  {label}: composite={r["composite_score"]:.3f}, ETA={r["eta_minutes"]} min')

In [ ]:
# Visualise on folium map
m = folium.Map(location=[ORIGIN['lat'], ORIGIN['lng']], zoom_start=13, tiles='CartoDB positron')

COLORS = ['#f59e0b', '#3b82f6', '#9ca3af']
LABELS = ['Driver Recommended', 'Fastest', 'Alternative']

for i, route in enumerate(ranked[:3]):
    geom = route_geometry(G, route['nodes'])
    # folium uses [lat, lng]
    latlngs = [[c[1], c[0]] for c in geom]
    folium.PolyLine(
        latlngs,
        color=COLORS[i],
        weight=5 if i == 0 else 3,
        opacity=1.0 if i == 0 else 0.6,
        tooltip=f'{LABELS[i]}: {route["eta_minutes"]} min, {route["distance_km"]} km',
    ).add_to(m)

folium.Marker([ORIGIN['lat'], ORIGIN['lng']], popup='Origin', icon=folium.Icon(color='green')).add_to(m)
folium.Marker([DEST['lat'],   DEST['lng']],   popup='Destination', icon=folium.Icon(color='red')).add_to(m)

m